# 04 — Gold Consolidation: Books → Consolidated

Consolidacao do Feature Store Gold via JOIN de 5 vias (LEFT JOIN) em `(NUM_CPF, SAFRA)`.

**Fontes**:
- Silver rawdata: `dados_cadastrais`, `telco`, `score_bureau_movel`
- Gold books: `book_recarga_cmv`, `book_pagamento`, `book_faturamento`

**Saida**: Gold `feature_store/clientes_consolidado/` (~400+ colunas, particionado por SAFRA)

**Hackathon PoD Academy** — Claro + Oracle

In [ ]:
import sys
from functools import reduce

sys.path.insert(0, "..")
from config import *

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, count as spark_count

print(f"NAMESPACE: {NAMESPACE}")
print(f"Silver bucket: {BUCKETS['silver']}")
print(f"Gold bucket: {BUCKETS['gold']}")
print(f"SAFRAS: {SAFRAS}")

In [ ]:
# Criar SparkSession com configuracao completa (Delta + Tuning)
builder = SparkSession.builder.appName("gold-consolidation")
for key, value in SPARK_CONFIG.items():
    builder = builder.config(key, value)
spark = builder.getOrCreate()

print(f"SparkSession criada: {spark.version}")

## Carregamento de Fontes

Carrega tabelas base do Silver rawdata e books do Gold.

In [ ]:
silver_bucket = BUCKETS["silver"]
gold_bucket = BUCKETS["gold"]

METADATA_COLS = ["_execution_id", "_data_inclusao", "_data_alteracao_silver", "DT_PROCESSAMENTO"]
KEY_COLS = ["NUM_CPF", "SAFRA"]
DUPLICATED_COLS = ["FLAG_INSTALACAO", "FPD", "PROD", "flag_mig2"]


def read_silver(spark, table_name):
    """Le tabela do Silver bucket."""
    uri = f"oci://{silver_bucket}@{NAMESPACE}/rawdata/{table_name}/"
    if SILVER_READ_FORMAT == "delta":
        return spark.read.format("delta").load(uri)
    return spark.read.parquet(uri)


def read_gold_book(spark, book_name):
    """Le book do Gold bucket."""
    uri = f"oci://{gold_bucket}@{NAMESPACE}/books/{book_name}/"
    return spark.read.format("delta").load(uri)


def drop_internal_cols(df):
    """Remove colunas internas de auditoria."""
    internal = ["_execution_id", "_data_inclusao", "_data_alteracao_silver"]
    for c in internal:
        if c in df.columns:
            df = df.drop(c)
    return df


def get_select_cols(df, alias, exclude, prefix=None):
    """Gera lista de colunas com renomeacao opcional por prefixo."""
    exclude_lower = {c.lower() for c in exclude}
    cols = []
    for c in df.columns:
        if c.lower() not in exclude_lower:
            if prefix:
                cols.append(f"{alias}.{c} AS {prefix}_{c}")
            else:
                cols.append(f"{alias}.{c}")
    return cols


# Carregar tabelas base do Silver
print("[GOLD] Carregando tabelas base do Silver...")
df_cadastro = drop_internal_cols(read_silver(spark, "dados_cadastrais"))
df_telco = drop_internal_cols(read_silver(spark, "telco"))
df_score = drop_internal_cols(read_silver(spark, "score_bureau_movel"))
print(f"  dados_cadastrais: {len(df_cadastro.columns)} cols")
print(f"  telco: {len(df_telco.columns)} cols")
print(f"  score_bureau_movel: {len(df_score.columns)} cols")

# Carregar books do Gold
print("\n[GOLD] Carregando books do Gold...")
df_book_rec = read_gold_book(spark, "book_recarga_cmv")
df_book_pag = read_gold_book(spark, "book_pagamento")
df_book_fat = read_gold_book(spark, "book_faturamento")
print(f"  book_recarga_cmv: {len(df_book_rec.columns)} cols")
print(f"  book_pagamento: {len(df_book_pag.columns)} cols")
print(f"  book_faturamento: {len(df_book_fat.columns)} cols")

# Registrar views
df_cadastro.createOrReplaceTempView("dados_cadastrais")
df_telco.createOrReplaceTempView("telco")
df_score.createOrReplaceTempView("score_bureau_movel")
df_book_rec.createOrReplaceTempView("book_recarga_cmv")
df_book_pag.createOrReplaceTempView("book_pagamento")
df_book_fat.createOrReplaceTempView("book_faturamento")

print("\n[OK] Todas as fontes carregadas como TempViews")

## Consolidacao do Feature Store

JOIN de 5 vias (LEFT JOIN) para criar o dataset consolidado `clientes_consolidado`:

```
dados_cadastrais (base)
  LEFT JOIN telco
  LEFT JOIN score_bureau_movel (-> TARGET_SCORE_01, TARGET_SCORE_02)
  LEFT JOIN book_recarga_cmv   (prefixo REC_)
  LEFT JOIN book_pagamento     (prefixo PAG_)
  LEFT JOIN book_faturamento   (prefixo FAT_, excluindo VLR_FPD por leakage)
```

Chave de JOIN: `(NUM_CPF, SAFRA)`

Escrita no Gold bucket em formato Delta, particionado por SAFRA.

In [ ]:
# Construir listas de colunas com prefixo
cols_cadastro = get_select_cols(df_cadastro, "c", METADATA_COLS)
cols_telco = get_select_cols(df_telco, "t", METADATA_COLS + KEY_COLS + DUPLICATED_COLS)

# Score: renomear SCORE_ para TARGET_SCORE_
exclude_score = METADATA_COLS + KEY_COLS + DUPLICATED_COLS
cols_score = []
for c in df_score.columns:
    if c.lower() not in {x.lower() for x in exclude_score}:
        if "SCORE_" in c:
            cols_score.append(f"s.{c} AS TARGET_{c}")
        else:
            cols_score.append(f"s.{c}")

cols_rec = get_select_cols(df_book_rec, "r", METADATA_COLS + KEY_COLS, prefix="REC")
cols_pag = get_select_cols(df_book_pag, "p2", METADATA_COLS + KEY_COLS, prefix="PAG")
cols_fat = get_select_cols(df_book_fat, "f2", METADATA_COLS + KEY_COLS + LEAKAGE_BLACKLIST, prefix="FAT")

all_cols = cols_cadastro + cols_telco + cols_score + cols_rec + cols_pag + cols_fat
all_cols.append("CURRENT_TIMESTAMP() AS DT_PROCESSAMENTO")

columns_sql = ",\n        ".join(all_cols)
safra_list = ", ".join(str(s) for s in SAFRAS)

query = f"""
SELECT
        {columns_sql}
FROM dados_cadastrais c
    LEFT JOIN telco t ON c.NUM_CPF = t.NUM_CPF AND c.SAFRA = t.SAFRA
    LEFT JOIN score_bureau_movel s ON c.NUM_CPF = s.NUM_CPF AND c.SAFRA = s.SAFRA
    LEFT JOIN book_recarga_cmv r ON c.NUM_CPF = r.NUM_CPF AND c.SAFRA = r.SAFRA
    LEFT JOIN book_pagamento p2 ON c.NUM_CPF = p2.NUM_CPF AND c.SAFRA = p2.SAFRA
    LEFT JOIN book_faturamento f2 ON c.NUM_CPF = f2.NUM_CPF AND c.SAFRA = f2.SAFRA
WHERE c.SAFRA IN ({safra_list})
"""

print(f"[GOLD] Executando query de consolidacao — {len(all_cols)} colunas esperadas")
df_gold = spark.sql(query)

# Escrever Gold em formato Delta
target_uri = GOLD_CONSOLIDATED_PATH
gold_writer = df_gold.write.mode("overwrite").partitionBy("SAFRA")
if STORAGE_FORMAT == "delta":
    gold_writer.format("delta").option("overwriteSchema", "true").save(target_uri)
else:
    gold_writer.parquet(target_uri)

total_rows = df_gold.count()
total_cols = len(df_gold.columns)
print(f"[GOLD] clientes_consolidado: {total_rows:,} linhas, {total_cols} colunas")
print(f"[GOLD] Target: {target_uri}")

## Validacao

Verificacoes criticas de integridade do Gold consolidado:

1. **Row count**: consistente com dados_cadastrais
2. **Anti-leakage**: `FAT_VLR_FPD` deve estar AUSENTE
3. **Targets presentes**: `TARGET_SCORE_01` e `TARGET_SCORE_02` devem existir
4. **SAFRAs completas**: 6 safras esperadas
5. **Sem duplicatas PK**: (NUM_CPF, SAFRA)
6. **Prefixos**: REC_, PAG_, FAT_ presentes

In [ ]:
# Validacao do Gold Consolidado
print(f"{'='*60}")
print(f"VALIDACAO — GOLD CONSOLIDATED")
print(f"{'='*60}")
checks_pass = 0
checks_fail = 0

# 1. Row count
cadastro_rows = df_cadastro.count()
if total_rows == cadastro_rows:
    print(f"  [PASS] Row count: {total_rows:,} == dados_cadastrais ({cadastro_rows:,})")
    checks_pass += 1
else:
    print(f"  [FAIL] Row count: {total_rows:,} != dados_cadastrais ({cadastro_rows:,})")
    checks_fail += 1

# 2. Anti-leakage: FAT_VLR_FPD deve estar ausente
leakage_found = [c for c in df_gold.columns if "VLR_FPD" in c]
if not leakage_found:
    print(f"  [PASS] Anti-leakage: nenhuma coluna VLR_FPD")
    checks_pass += 1
else:
    print(f"  [FAIL] Anti-leakage: encontradas {leakage_found}")
    checks_fail += 1

# 3. Targets presentes
required = REQUIRED_COLUMNS
missing = [c for c in required if c not in df_gold.columns]
if not missing:
    print(f"  [PASS] Colunas obrigatorias presentes: {required}")
    checks_pass += 1
else:
    print(f"  [FAIL] Colunas ausentes: {missing}")
    checks_fail += 1

# 4. SAFRAs completas
safra_count = df_gold.select("SAFRA").distinct().count()
if safra_count == len(SAFRAS):
    print(f"  [PASS] SAFRAs: {safra_count} safras")
    checks_pass += 1
else:
    print(f"  [FAIL] SAFRAs: {safra_count} (esperado: {len(SAFRAS)})")
    checks_fail += 1

# 5. PK uniqueness
dup_count = df_gold.groupBy("NUM_CPF", "SAFRA").agg(spark_count("*").alias("cnt")).filter(col("cnt") > 1).count()
if dup_count == 0:
    print(f"  [PASS] Sem duplicatas PK (NUM_CPF, SAFRA)")
    checks_pass += 1
else:
    print(f"  [FAIL] {dup_count:,} duplicatas PK")
    checks_fail += 1

# 6. Prefixos
for prefix in EXPECTED_PREFIXES:
    prefix_cols = [c for c in df_gold.columns if c.startswith(prefix)]
    if len(prefix_cols) > 0:
        print(f"  [PASS] Prefixo {prefix}: {len(prefix_cols)} colunas")
        checks_pass += 1
    else:
        print(f"  [FAIL] Prefixo {prefix}: nenhuma coluna")
        checks_fail += 1

print(f"\n  Resultado: {checks_pass} PASS / {checks_fail} FAIL")
print(f"{'='*60}")

print(f"\n[GOLD] CONSOLIDACAO COMPLETA")
print(f"  Consolidado: {total_rows:,} linhas, {total_cols} colunas")
print(f"  SAFRAs: {safra_count}")
print(f"  Formato: {STORAGE_FORMAT}")
print(f"  Target: {target_uri}")